# PCA 1대1 실험 (Train/Val 기반 선택) — 화학 X → 관능 Y

- 데이터: `/home/a202192020/맥주데이터실험/data/Supplemental Files and Figure source files.xlsx`
- 출력: `/home/a202192020/맥주데이터실험/pca/output/... (타임스탬프 폴더 자동 생성)`
- 핵심 아이디어  
  - **선택/선별은 train/val에서만** 수행  
  - test는 **최종 평가용**(원칙)  
  - PCA는 **X에 대해서만 1번 fit** → 모든 타깃(Y)에서 동일한 변환을 재사용  
  - 변환된 PCA 차원(`Z`)을 **.npy로 저장**해서 재실행 시 재사용 가능

> 주의: test까지 사용해서 “선별”하면 test가 더 이상 순수 평가셋이 아니게 됩니다.  
> 이 노트북은 원칙적으로 test를 선별에 사용하지 않도록 작성했습니다(대신 train/val 갭으로 과적합 필터).

In [ ]:
# ===== (필요시) 라이브러리 일괄 설치 =====
# - 서버 환경에 따라 이미 설치되어 있으면 빠르게 넘어갑니다.
!python -m pip install -q --upgrade pip
!python -m pip install -q pandas numpy scikit-learn openpyxl joblib tqdm matplotlib

In [ ]:
import os, time, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

from joblib import dump
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
# ===== 설정 =====
XLSX_PATH = r"/home/a202192020/맥주데이터실험/data/Supplemental Files and Figure source files.xlsx"
OUTPUT_ROOT = r"/home/a202192020/맥주데이터실험/pca/output"

SEED = 42

# outer split(논문): 70/30 = 175/75, style stratify
TEST_SIZE = 0.30

# inner split: train -> train_fit/val (선별용)
VAL_SIZE = 0.20

# PCA 후보 k (train_fit 샘플 수에 따라 자동으로 max_k가 달라짐)
# - 아래 리스트는 기본 후보이고, 마지막에 자동으로 max_k를 추가합니다.
K_LIST_BASE = [1, 2, 4, 8, 16, 32, 64, 128]

# 과적합(Overfitting) 필터: train과 val 사이의 R² 차이가 너무 크면 제외
MAX_TRAIN_VAL_GAP_R2 = 0.15

# "좋은 조합"으로 볼 최소 val 성능 (필요하면 조정)
MIN_VAL_R2 = 0.20

# GBR(Gradient Boosting Regressor) 파라미터 (논문 스타일의 트리 기반 모델)
# - 논문은 그리드서치로 최적화했지만, 1대1 전수조사에서는 계산 폭발 방지를 위해 고정 파라미터를 기본으로 둡니다.
GBR_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    random_state=SEED,
)

In [ ]:
# ===== 유틸 =====
def set_korea_timezone():
    # 리눅스에서 TZ를 서울로 맞춤 (가능하면)
    os.environ["TZ"] = "Asia/Seoul"
    try:
        time.tzset()
    except Exception:
        pass

def make_run_dir(root: str, prefix: str) -> Path:
    set_korea_timezone()
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = Path(root) / f"{prefix}_{ts}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir

def rmse(y_true, y_pred) -> float:
    # sklearn 버전 호환을 위해 squared=False 대신 직접 sqrt
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def make_stratify_labels(styles: pd.Series, min_count: int = 2, rare_label: str = "RARE") -> np.ndarray:
    vc = styles.value_counts()
    return styles.map(lambda s: s if vc.get(s, 0) >= min_count else rare_label).astype(str).values

def safe_train_test_split(X, y, test_size, seed, stratify_labels=None):
    # stratify 실패 시 fallback으로 stratify 없이 분할
    try:
        return train_test_split(X, y, test_size=test_size, random_state=seed, shuffle=True, stratify=stratify_labels)
    except Exception as e:
        print("[WARN] stratified split failed -> fallback to non-stratified split")
        print("       reason:", repr(e))
        return train_test_split(X, y, test_size=test_size, random_state=seed, shuffle=True, stratify=None)

In [ ]:
# ===== 데이터 로드 (S1: 화학 X, S4: 관능 Y) =====
assert Path(XLSX_PATH).exists(), f"File not found: {XLSX_PATH}"

s1 = pd.read_excel(XLSX_PATH, sheet_name="Supplementary File S1")
s4 = pd.read_excel(XLSX_PATH, sheet_name="Supplementary File S4")

# 컬럼 정의
meta_cols = ["beer", "beer_id", "tasting_category_fine"]
X_cols = [c for c in s1.columns if c not in meta_cols]
Y_cols = [c for c in s4.columns if c not in meta_cols]

df = s1[meta_cols + X_cols].merge(
    s4[["beer_id"] + Y_cols],
    on="beer_id",
    how="inner",
    validate="one_to_one",
)

print("df shape:", df.shape)
print("X dim:", len(X_cols), "Y dim:", len(Y_cols))
print("styles:", df["tasting_category_fine"].nunique())

# 결측 확인
missing_x = df[X_cols].isna().sum().sum()
missing_y = df[Y_cols].isna().sum().sum()
print("missing in X:", int(missing_x), "| missing in Y:", int(missing_y))

display(df.head())

In [ ]:
# ===== Outer split (논문 스타일): 70/30 + style stratify =====
y_style = df["tasting_category_fine"].astype(str)

idx = np.arange(len(df))
idx_train, idx_test = train_test_split(
    idx,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
    stratify=y_style,
)

df_train = df.iloc[idx_train].reset_index(drop=True)
df_test  = df.iloc[idx_test].reset_index(drop=True)

print("train:", df_train.shape, "test:", df_test.shape)
print("\n[train style counts]\n", df_train["tasting_category_fine"].value_counts().sort_values().head(10))
print("\n[test style counts]\n", df_test["tasting_category_fine"].value_counts().sort_values().head(10))

In [ ]:
# ===== Preprocess (mean imputation -> standardization) + PCA fit on OUTER TRAIN (최종 모델용 캐시) =====
run_dir = make_run_dir(OUTPUT_ROOT, prefix="pca1to1_trainval")
print("RUN_DIR =", run_dir)

# X/Y numpy
X_train = df_train[X_cols].to_numpy(dtype=float)
X_test  = df_test[X_cols].to_numpy(dtype=float)

Y_train = df_train[Y_cols].to_numpy(dtype=float)
Y_test  = df_test[Y_cols].to_numpy(dtype=float)

# preprocess fit on outer train
imp_full = SimpleImputer(strategy="mean")
sc_full = StandardScaler()

X_train_imp = imp_full.fit_transform(X_train)
X_train_sc  = sc_full.fit_transform(X_train_imp)

X_test_imp = imp_full.transform(X_test)
X_test_sc  = sc_full.transform(X_test_imp)

# PCA fit on outer train (max components = min(n-1, d))
max_k_full = int(min(X_train_sc.shape[0] - 1, X_train_sc.shape[1]))
pca_full = PCA(n_components=max_k_full, random_state=SEED)
Z_train_full = pca_full.fit_transform(X_train_sc)
Z_test_full  = pca_full.transform(X_test_sc)

print("X_train_sc:", X_train_sc.shape, "| Z_train_full:", Z_train_full.shape, "| max_k_full:", max_k_full)

# 저장 (재사용 목적)
dump(imp_full, run_dir / "imputer_outer_train.joblib")
dump(sc_full,  run_dir / "scaler_outer_train.joblib")
dump(pca_full, run_dir / "pca_outer_train.joblib")

np.save(run_dir / "X_train_scaled.npy", X_train_sc)
np.save(run_dir / "X_test_scaled.npy",  X_test_sc)
np.save(run_dir / "Z_train_full.npy", Z_train_full)
np.save(run_dir / "Z_test_full.npy",  Z_test_full)

with open(run_dir / "columns.json", "w", encoding="utf-8") as f:
    json.dump({"X_cols": X_cols, "Y_cols": Y_cols}, f, ensure_ascii=False, indent=2)

with open(run_dir / "outer_split_indices.json", "w", encoding="utf-8") as f:
    json.dump({"idx_train": idx_train.tolist(), "idx_test": idx_test.tolist()}, f, ensure_ascii=False, indent=2)

print("Saved caches to:", run_dir)

In [ ]:
# ===== Inner split: OUTER TRAIN -> train_fit / val (선별용) =====
style_train = df_train["tasting_category_fine"].astype(str)

# stratify가 실패할 가능성을 줄이기 위해(특히 희소 스타일) RARE merge 라벨 사용
strat_labels = make_stratify_labels(style_train, min_count=2, rare_label="RARE")

idx_inner = np.arange(len(df_train))
idx_fit, idx_val = train_test_split(
    idx_inner,
    test_size=VAL_SIZE,
    random_state=SEED,
    shuffle=True,
    stratify=strat_labels,
)

df_fit = df_train.iloc[idx_fit].reset_index(drop=True)
df_val = df_train.iloc[idx_val].reset_index(drop=True)

print("train_fit:", df_fit.shape, "val:", df_val.shape)
print("min style count in train_fit:", int(df_fit["tasting_category_fine"].value_counts().min()))

In [ ]:
# ===== Preprocess + PCA fit on train_fit (선별 공정성: val을 보지 않음) =====
X_fit = df_fit[X_cols].to_numpy(dtype=float)
X_val = df_val[X_cols].to_numpy(dtype=float)

Y_fit = df_fit[Y_cols].to_numpy(dtype=float)
Y_val = df_val[Y_cols].to_numpy(dtype=float)

imp = SimpleImputer(strategy="mean")
sc  = StandardScaler()

X_fit_imp = imp.fit_transform(X_fit)
X_fit_sc  = sc.fit_transform(X_fit_imp)

X_val_imp = imp.transform(X_val)
X_val_sc  = sc.transform(X_val_imp)

max_k_fit = int(min(X_fit_sc.shape[0] - 1, X_fit_sc.shape[1]))
pca = PCA(n_components=max_k_fit, random_state=SEED)
Z_fit_full = pca.fit_transform(X_fit_sc)
Z_val_full = pca.transform(X_val_sc)

# k 후보 리스트 구성 (max_k_fit를 자동 포함)
K_LIST = [k for k in K_LIST_BASE if k <= max_k_fit]
if max_k_fit not in K_LIST:
    K_LIST.append(max_k_fit)

print("max_k_fit:", max_k_fit)
print("K_LIST:", K_LIST)

# 저장
dump(imp, run_dir / "imputer_trainfit.joblib")
dump(sc,  run_dir / "scaler_trainfit.joblib")
dump(pca, run_dir / "pca_trainfit.joblib")

np.save(run_dir / "X_fit_scaled.npy", X_fit_sc)
np.save(run_dir / "X_val_scaled.npy", X_val_sc)
np.save(run_dir / "Z_fit_full.npy", Z_fit_full)
np.save(run_dir / "Z_val_full.npy", Z_val_full)

with open(run_dir / "inner_split_indices.json", "w", encoding="utf-8") as f:
    json.dump({"idx_fit": idx_fit.tolist(), "idx_val": idx_val.tolist()}, f, ensure_ascii=False, indent=2)

print("Saved inner caches to:", run_dir)

In [ ]:
# ===== 1대1: 각 관능(Y_j)별로 PCA 차원 k를 train/val로 선택 =====
def build_gbr():
    # sklearn 구버전 호환: loss 이름 대응
    params = dict(GBR_PARAMS)
    loss = params.pop("loss", "squared_error")
    try:
        _ = GradientBoostingRegressor(loss=loss, **params)
    except TypeError:
        # 매우 구버전이면 loss 파라미터 자체가 다를 수 있음 -> 기본 loss 사용
        _ = GradientBoostingRegressor(**params)
        loss = None
    if loss is None:
        return _
    # loss 이름 호환
    if loss == "squared_error":
        try:
            return GradientBoostingRegressor(loss="squared_error", **params)
        except Exception:
            return GradientBoostingRegressor(loss="ls", **params)
    if loss == "absolute_error":
        try:
            return GradientBoostingRegressor(loss="absolute_error", **params)
        except Exception:
            return GradientBoostingRegressor(loss="lad", **params)
    return GradientBoostingRegressor(loss=loss, **params)

results_long = []
best_rows = []

for j, y_name in enumerate(tqdm(Y_cols, desc="Targets (Y)")):
    y_tr = Y_fit[:, j]
    y_va = Y_val[:, j]

    best = None  # (val_r2, k, rowdict)

    for k in K_LIST:
        Xtr_k = Z_fit_full[:, :k]
        Xva_k = Z_val_full[:, :k]

        model = build_gbr()
        model.fit(Xtr_k, y_tr)

        pred_tr = model.predict(Xtr_k)
        pred_va = model.predict(Xva_k)

        r2_tr = float(r2_score(y_tr, pred_tr))
        r2_va = float(r2_score(y_va, pred_va))
        gap = r2_tr - r2_va

        row = dict(
            target=y_name,
            k=int(k),
            r2_train=r2_tr,
            r2_val=r2_va,
            rmse_train=rmse(y_tr, pred_tr),
            rmse_val=rmse(y_va, pred_va),
            gap_train_val=gap,
        )
        results_long.append(row)

        # 선택 규칙: val R2 최대 + 과적합(gap) 제한
        ok = (r2_va >= MIN_VAL_R2) and (gap <= MAX_TRAIN_VAL_GAP_R2)
        if ok:
            if (best is None) or (r2_va > best[0]):
                best = (r2_va, k, row)

    # 만약 조건을 만족하는 k가 없으면, 그냥 val R2가 최대인 k 선택
    if best is None:
        # val R2 최대 선택
        best_any = max([r for r in results_long if r["target"] == y_name], key=lambda r: r["r2_val"])
        best_rows.append({**best_any, "selected_by": "best_val_no_filter"})
    else:
        best_rows.append({**best[2], "selected_by": "best_val_with_gap_filter"})

df_long = pd.DataFrame(results_long)
df_best = pd.DataFrame(best_rows)

df_long.to_csv(run_dir / "per_target_per_k_trainval.csv", index=False, encoding="utf-8-sig")
df_best.to_csv(run_dir / "best_k_by_trainval.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print(" -", run_dir / "per_target_per_k_trainval.csv")
print(" -", run_dir / "best_k_by_trainval.csv")

display(df_best.sort_values("r2_val", ascending=False).head(10))

In [ ]:
# ===== 선택된 k로 OUTER TRAIN 전체로 학습 후 TEST 평가 =====
# (주의) 여기서는 PCA를 outer train 기준으로 fit한 pca_full을 사용합니다. (평가 공정성: test는 전혀 사용하지 않음)
df_best = pd.read_csv(run_dir / "best_k_by_trainval.csv")

final_rows = []
models_dir = run_dir / "models_final"
models_dir.mkdir(exist_ok=True)

for _, row in tqdm(df_best.iterrows(), total=len(df_best), desc="Final train->test"):
    y_name = row["target"]
    k = int(row["k"])
    j = Y_cols.index(y_name)

    y_tr = Y_train[:, j]
    y_te = Y_test[:, j]

    Xtr_k = Z_train_full[:, :k]
    Xte_k = Z_test_full[:, :k]

    model = build_gbr()
    model.fit(Xtr_k, y_tr)

    pred_tr = model.predict(Xtr_k)
    pred_te = model.predict(Xte_k)

    out = dict(
        target=y_name,
        k=k,
        r2_train=float(r2_score(y_tr, pred_tr)),
        r2_test=float(r2_score(y_te, pred_te)),
        rmse_train=rmse(y_tr, pred_tr),
        rmse_test=rmse(y_te, pred_te),
        selected_by=row.get("selected_by", ""),
        r2_val=float(row["r2_val"]),
        gap_train_val=float(row["gap_train_val"]),
    )
    final_rows.append(out)

    # 모델 저장(선택사항)
    dump(model, models_dir / f"gbr_{y_name}_k{k}.joblib")

df_final = pd.DataFrame(final_rows)
df_final.to_csv(run_dir / "final_test_metrics.csv", index=False, encoding="utf-8-sig")

print("Saved final metrics:", run_dir / "final_test_metrics.csv")
display(df_final.sort_values("r2_test", ascending=False).head(10))